In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [3]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_ollama import ChatOllama
model = ChatOllama(model="gemma4:e2b")

agent = create_agent(
    model,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [6]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='40558243-ba2a-42c3-bf95-8bd9bfa6c9f1'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-04-22T15:38:42.3375627Z', 'done': True, 'done_reason': 'stop', 'total_duration': 46232053800, 'load_duration': 16642111700, 'prompt_eval_count': 84, 'prompt_eval_duration': 6349366500, 'eval_count': 214, 'eval_duration': 22188382600, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--019db5d7-406e-76b3-aaea-150e1e1d3bdd-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'bb100e7e-5a12-474c-b97e-95a53851aba3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 84, 'output_tokens': 214, 'total_tokens': 298}),
              ToolMessage(content='Successfully updated favourite colour'

In [7]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='c8107ece-f54d-49a6-b034-b494fb482159'),
              AIMessage(content='Hello! I am doing well, thank you for asking. How can I help you today?', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-04-22T15:39:03.3636838Z', 'done': True, 'done_reason': 'stop', 'total_duration': 18824908300, 'load_duration': 593839300, 'prompt_eval_count': 85, 'prompt_eval_duration': 439311600, 'eval_count': 172, 'eval_duration': 17598360200, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--019db5d7-fdba-7e80-888d-cd1f92f4ff47-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 85, 'output_tokens': 172, 'total_tokens': 257})]}


## Read state

In [8]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    model,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [9]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='78f34771-6651-42e1-81f9-c9b2d44d6558'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-04-22T15:39:29.7170784Z', 'done': True, 'done_reason': 'stop', 'total_duration': 26234245300, 'load_duration': 510156900, 'prompt_eval_count': 119, 'prompt_eval_duration': 1913578500, 'eval_count': 228, 'eval_duration': 23529846000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--019db5d8-47b8-7f80-b6b8-108f0ccaacf5-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'dbd37715-ad8d-4dcc-bc19-3db8ce68f9b2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 119, 'output_tokens': 228, 'total_tokens': 347}),
              ToolMessage(content='Successfully updated favourite colour'

In [10]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='78f34771-6651-42e1-81f9-c9b2d44d6558'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-04-22T15:39:29.7170784Z', 'done': True, 'done_reason': 'stop', 'total_duration': 26234245300, 'load_duration': 510156900, 'prompt_eval_count': 119, 'prompt_eval_duration': 1913578500, 'eval_count': 228, 'eval_duration': 23529846000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--019db5d8-47b8-7f80-b6b8-108f0ccaacf5-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'dbd37715-ad8d-4dcc-bc19-3db8ce68f9b2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 119, 'output_tokens': 228, 'total_tokens': 347}),
              ToolMessage(content='Successfully updated favourite colour'